# Reentrenamiento YOLO26n con particiones sin duplicados · AMIR · DSI EVAL-02
Responde al comentario del profe: *"Depuren duplicados antes de fijar las particiones"*.
1. Junta todas las imágenes de Hard Hats v6 (train + valid + test del zip original).
2. Calcula la huella 16×16 (la misma del notebook Eval 03) y agrupa las imágenes idénticas o casi idénticas (distancia ≤ 12 de 256 bits).
3. Reparte **grupos completos** (no imágenes sueltas) a train/valid/test, con las proporciones originales. Así ninguna escena aparece en dos conjuntos.
4. Reentrena YOLO26n con los mismos ajustes del entrenamiento original (640 px, hasta 30 épocas, batch 16, AdamW, paciencia 10).
5. Evalúa en el nuevo Test e imprime el inventario de duplicados.


In [1]:
import importlib.util
if importlib.util.find_spec("ultralytics") is None:
    get_ipython().run_line_magic("pip", "install -q ultralytics")
import os, re, glob, json, zipfile, shutil, time, random, warnings, yaml
import numpy as np, pandas as pd
from PIL import Image
import torch
warnings.filterwarnings("ignore")
DATASET_ZIP = "/content/drive/MyDrive/AMIR/DATASET_ZIP/Hard Hats.v6-resized640_noaugmentation-rf-detr.yolo26.zip"
SALIDA = "/content/drive/MyDrive/entrenamiento_casco_sin_duplicados"
NOMBRE_CORRIDA = "yolo26n_hardhat_sin_dup"
UMBRAL_CASI = 12       # igual que Eval 03 (de 256 bits)
SEED = 42
EPOCAS, IMGSZ, BATCH, PACIENCIA = 30, 640, 16, 10
if not os.path.exists(DATASET_ZIP):
    from google.colab import drive; drive.mount("/content/drive")
assert os.path.exists(DATASET_ZIP), "No encuentro el zip del dataset."
os.makedirs(SALIDA, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == 0 else "NO HAY GPU: activa una T4")

Mounted at /content/drive
GPU: Tesla T4


## 1 · Juntar las imágenes originales

In [2]:
ORIG = "/content/original"
if not os.path.exists(ORIG + "/.listo"):
    shutil.rmtree(ORIG, ignore_errors=True); zipfile.ZipFile(DATASET_ZIP).extractall(ORIG); open(ORIG + "/.listo", "w").write("ok")
EXT = (".jpg", ".jpeg", ".png")
filas = []
for d in glob.glob(ORIG + "/**/images", recursive=True):
    split = os.path.basename(os.path.dirname(d)); split = "valid" if split == "val" else split
    for f in sorted(os.listdir(d)):
        if f.lower().endswith(EXT):
            lab = os.path.join(os.path.dirname(d), "labels", os.path.splitext(f)[0] + ".txt")
            filas.append({"split_original": split, "ruta": os.path.join(d, f), "etiqueta": lab if os.path.exists(lab) else ""})
POOL = pd.DataFrame(filas)
print(POOL.split_original.value_counts().to_string()); print("Total:", len(POOL), "| sin archivo de etiquetas:", int((POOL.etiqueta == "").sum()))
_y = [p for p in glob.glob(ORIG + "/**/data.yaml", recursive=True)][0]
NAMES = yaml.safe_load(open(_y))["names"]; print("Clases:", NAMES)

split_original
train    13782
valid     3962
test      2001
Total: 19745 | sin archivo de etiquetas: 0
Clases: ['Hardhat', 'NO-Hardhat']


## 2 · Huellas y grupos de duplicados

In [3]:
def huella(r):
    with Image.open(r) as im: a = np.asarray(im.convert("L").resize((16, 16), Image.LANCZOS), dtype=np.float32)
    return (a > a.mean()).flatten()
H = np.stack([huella(r) for r in POOL.ruta]).astype(np.float32)

padre = list(range(len(H)))
def raiz(i):
    while padre[i] != i: padre[i] = padre[padre[i]]; i = padre[i]
    return i
def unir(a, b):
    ra, rb = raiz(a), raiz(b)
    if ra != rb: padre[rb] = ra
pares = 0
for k in range(0, len(H), 500):
    A = H[k:k + 500]; D = 256 - (A @ H.T + (1 - A) @ (1 - H).T)
    for i, j in zip(*np.where(D <= UMBRAL_CASI)):
        if k + i < j: unir(k + i, j); pares += 1
POOL["grupo"] = [raiz(i) for i in range(len(H))]
tam = POOL.groupby("grupo").size()
print(f"Pares casi idénticos: {pares} | grupos: {len(tam)} | grupos con más de 1 imagen: {(tam > 1).sum()} | imágenes en esos grupos: {int(tam[tam > 1].sum())}")

Pares casi idénticos: 99488 | grupos: 15384 | grupos con más de 1 imagen: 1538 | imágenes en esos grupos: 5899


## 3 · Nueva partición por grupos


In [4]:
prop = POOL.split_original.value_counts(normalize=True)
objetivo = {s: prop.get(s, 0) * len(POOL) for s in ("train", "valid", "test")}
rng = random.Random(SEED); grupos = list(tam.index); rng.shuffle(grupos)
asig = {}; cuenta = {s: 0 for s in objetivo}
# los grupos grandes primero (mejor encaje), el resto al azar
grandes = [g for g in grupos if tam[g] > 1]; chicos = [g for g in grupos if tam[g] == 1]
for g in sorted(grandes, key=lambda g: -tam[g]) + chicos:
    s = max(objetivo, key=lambda s: objetivo[s] - cuenta[s])      # el conjunto al que más le falta
    asig[g] = s; cuenta[s] += tam[g]
POOL["split_nuevo"] = POOL.grupo.map(asig)
print(pd.crosstab(POOL.split_original, POOL.split_nuevo))
# verificación: ningún par casi idéntico entre conjuntos distintos
malos = 0
for k in range(0, len(H), 500):
    A = H[k:k + 500]; D = 256 - (A @ H.T + (1 - A) @ (1 - H).T)
    for i, j in zip(*np.where(D <= UMBRAL_CASI)):
        if POOL.split_nuevo.iloc[k + i] != POOL.split_nuevo.iloc[j]: malos += 1
print("Pares casi idénticos entre conjuntos distintos tras la nueva partición:", malos)
assert malos == 0
POOL[["ruta", "split_original", "split_nuevo", "grupo"]].assign(imagen=lambda d: d.ruta.map(os.path.basename)).drop(columns="ruta") \
    .to_csv(os.path.join(SALIDA, "inventario_duplicados_y_particion.csv"), index=False)
print("Inventario guardado en Drive.")

split_nuevo     test  train  valid
split_original                    
test             217   1375    409
train           1377   9699   2706
valid            407   2708    847
Pares casi idénticos entre conjuntos distintos tras la nueva partición: 0
Inventario guardado en Drive.


In [5]:
NUEVO = "/content/dataset_sin_dup"; shutil.rmtree(NUEVO, ignore_errors=True)
for s in ("train", "valid", "test"):
    os.makedirs(f"{NUEVO}/{s}/images"); os.makedirs(f"{NUEVO}/{s}/labels")
for _, r in POOL.iterrows():
    s = r.split_nuevo; n = f"{r.split_original}_{os.path.basename(r.ruta)}"          # prefijo evita nombres repetidos
    shutil.copy(r.ruta, f"{NUEVO}/{s}/images/{n}")
    if r.etiqueta: shutil.copy(r.etiqueta, f"{NUEVO}/{s}/labels/{os.path.splitext(n)[0]}.txt")
DATA_YAML = os.path.join(SALIDA, "dataset_sin_dup.yaml")
yaml.safe_dump({"path": NUEVO, "train": "train/images", "val": "valid/images", "test": "test/images", "names": NAMES}, open(DATA_YAML, "w"))
# distribución de clases por conjunto (para ver que ninguna quedó vacía o muy desbalanceada)
cont = {}
for s in ("train", "valid", "test"):
    c = {}
    for f in glob.glob(f"{NUEVO}/{s}/labels/*.txt"):
        for l in open(f):
            if l.strip(): k = int(float(l.split()[0])); c[k] = c.get(k, 0) + 1
    cont[s] = c
print(pd.DataFrame(cont).rename(index={k: (NAMES[k] if isinstance(NAMES, list) else NAMES[k]) for k in range(len(NAMES))}))

            train  valid  test
Hardhat     27957   9482  4989
NO-Hardhat   9599   2120  1246


## 4 · Entrenamiento (mismos ajustes que el original)


In [6]:
from ultralytics import YOLO
RESUMIR = True
CARPETA = os.path.join(SALIDA, NOMBRE_CORRIDA); ULTIMO = os.path.join(CARPETA, "weights", "last.pt")
t0 = time.time()
if RESUMIR and os.path.exists(ULTIMO):
    modelo = YOLO(ULTIMO); modelo.train(resume=True)
else:
    modelo = YOLO("yolo26n.pt")
    modelo.train(data=DATA_YAML, epochs=EPOCAS, imgsz=IMGSZ, batch=BATCH, patience=PACIENCIA, seed=SEED,
                 project=SALIDA, name=NOMBRE_CORRIDA, exist_ok=True, device=DEVICE, workers=2, plots=True)
print(f"Entrenamiento: {(time.time() - t0) / 60:.1f} min (esta ejecución)")
MEJOR = os.path.join(CARPETA, "weights", "best.pt")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/entrenamiento_casco_sin_duplicados/dataset_sin_dup.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5

## 5 · Evaluación en el nuevo Test (sin duplicados con Train)

In [7]:
m = YOLO(MEJOR); r = m.val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=True, verbose=False)
TAB = pd.DataFrame({"clase": [m.names[i] for i in r.box.ap_class_index], "P": r.box.p, "R": r.box.r, "mAP50": r.box.ap50, "mAP50-95": r.box.ap}).round(3)
GLOBAL = {"P": round(float(r.box.mp), 3), "R": round(float(r.box.mr), 3), "mAP50": round(float(r.box.map50), 3), "mAP50-95": round(float(r.box.map), 3)}
print("Test sin duplicados:", GLOBAL); display(TAB)
json.dump({"global": GLOBAL, "por_clase": TAB.to_dict("records")}, open(os.path.join(SALIDA, "metricas_test_sin_duplicados.json"), "w"), ensure_ascii=False, indent=1)
FINAL = os.path.join(SALIDA, "modelo_casco_yolo26_sin_duplicados.pt"); shutil.copy(MEJOR, FINAL); print("Pesos:", FINAL)

Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,389,978 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 125.3±224.2 MB/s, size: 61.4 KB)
val: Scanning /content/dataset_sin_dup/test/labels... 2001 images, 30 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2001/2001 450.1it/s 4.4s
val: New cache created: /content/dataset_sin_dup/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 126/126 6.0it/s 20.9s
                   all       2001       6235      0.915      0.903      0.945      0.602
Speed: 1.1ms preprocess, 3.7ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to /content/runs/detect/val
Test sin duplicados: {'P': 0.915, 'R': 0.903, 'mAP50': 0.945, 'mAP50-95': 0.602}


,clase,P,R,mAP50,mAP50-95
0,Hardhat,0.913,0.919,0.949,0.579
1,NO-Hardhat,0.917,0.887,0.940,0.626


Pesos: /content/drive/MyDrive/entrenamiento_casco_sin_duplicados/modelo_casco_yolo26_sin_duplicados.pt
